> **Chapter 11, Part 4** | Enterprise case study. **Focus:** duplicate clusters, threshold sensitivity, and when stewardship should interrupt automatic merge logic.


# Enterprise Case Study: Duplicate Clusters and Stewardship Risk

This notebook is the concrete bridge the chapter needs.

The fractal analogy is useful because it sharpens our attention to scale and boundary sensitivity. The MDM problem here is not fractal geometry in a strict sense. It is that a very small threshold change can alter entity boundaries in ways that matter operationally. When that happens, the cluster should not be treated as routine. It should be treated as a stewardship candidate.

## Outputs

- a synthetic duplicate-resolution dataset
- a threshold-sensitive clustering exercise
- a simple instability score for stewardship triage
- a practical reading of where automatic merge logic should stop

## Supporting reading

- IBM on master data management: https://www.ibm.com/think/topics/master-data-management
- Abraham, Schneider, and vom Brocke (2019): https://link.springer.com/article/10.1007/s12599-019-00588-3
- Song, Havlin, and Makse on self-similar networks: https://www.nature.com/articles/nature03248

## Failure note

If the threshold moves by two points and the entity boundary flips, calling the result a stable golden record is careless.

## How I would debug this

I would inspect the ambiguous cluster first, not the clean one. The point is to find the records whose identity depends on the threshold rather than pretending the threshold is neutral.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

records = pd.DataFrame(
    [
        {"record_id": "R1", "source": "crm", "name": "Acme Health", "city": "Chicago", "email": "ops@acmehealth.com"},
        {"record_id": "R2", "source": "erp", "name": "ACME Health Inc", "city": "Chicago", "email": "operations@acmehealth.com"},
        {"record_id": "R3", "source": "support", "name": "Acme Health", "city": "Skokie", "email": "ops@acmehealth.com"},
        {"record_id": "R4", "source": "crm", "name": "Northwind Labs", "city": "Boston", "email": "data@northwindlabs.com"},
        {"record_id": "R5", "source": "erp", "name": "Northwind Laboratories", "city": "Boston", "email": "data@northwindlabs.com"},
        {"record_id": "R6", "source": "partner", "name": "North Wind Labs", "city": "Cambridge", "email": "contact@northwindlabs.com"},
        {"record_id": "R7", "source": "crm", "name": "Riverstone Foods", "city": "Austin", "email": "hello@riverstone.com"},
        {"record_id": "R8", "source": "support", "name": "River Stone Food Group", "city": "Austin", "email": "info@riverstonefoods.com"},
    ]
)

scores = pd.DataFrame(
    [
        ("R1", "R2", 0.94),
        ("R1", "R3", 0.89),
        ("R2", "R3", 0.85),
        ("R4", "R5", 0.96),
        ("R4", "R6", 0.84),
        ("R5", "R6", 0.82),
        ("R7", "R8", 0.87),
        ("R2", "R6", 0.32),
        ("R3", "R4", 0.18),
    ],
    columns=["left", "right", "score"],
)

records


In [ ]:
def clusters_at_threshold(score_frame, threshold, ids):
    parent = {record_id: record_id for record_id in ids}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for row in score_frame.itertuples(index=False):
        if row.score >= threshold:
            union(row.left, row.right)

    clusters = {}
    for record_id in ids:
        root = find(record_id)
        clusters.setdefault(root, []).append(record_id)

    cluster_map = {}
    for cluster_id, members in enumerate(clusters.values(), start=1):
        for member in members:
            cluster_map[member] = f"C{cluster_id}"

    return cluster_map, list(clusters.values())


def membership_signature(score_frame, thresholds, ids):
    signatures = {record_id: [] for record_id in ids}
    cluster_counts = []

    for threshold in thresholds:
        assignment, clusters = clusters_at_threshold(score_frame, threshold, ids)
        cluster_counts.append(
            {
                "threshold": threshold,
                "cluster_count": len(clusters),
                "largest_cluster": max(len(cluster) for cluster in clusters),
            }
        )
        for record_id in ids:
            signatures[record_id].append(assignment[record_id])

    return pd.DataFrame(cluster_counts), signatures


thresholds = [0.82, 0.84, 0.86, 0.88, 0.90, 0.94]
summary, signatures = membership_signature(scores, thresholds, records["record_id"].tolist())
summary


In [ ]:
instability = pd.DataFrame(
    [
        {
            "record_id": record_id,
            "cluster_path": " -> ".join(signature),
            "distinct_assignments": len(set(signature)),
            "stewardship_flag": len(set(signature)) > 2,
        }
        for record_id, signature in signatures.items()
    ]
).merge(records, on="record_id")

instability.sort_values(["stewardship_flag", "distinct_assignments"], ascending=[False, False])


In [ ]:
plt.figure(figsize=(8, 4.8))
plt.plot(summary["threshold"], summary["cluster_count"], marker="o", label="number of clusters")
plt.plot(summary["threshold"], summary["largest_cluster"], marker="s", label="largest cluster size")
plt.gca().invert_xaxis()
plt.xlabel("match threshold")
plt.ylabel("count")
plt.title("Duplicate cluster behavior across thresholds")
plt.legend()
plt.tight_layout()
plt.show()


## Reading the case

`R1`, `R2`, and `R3` form a cluster that looks plausible at first glance. The problem is that the membership path changes across nearby thresholds. That is a boundary-sensitive region. `R4`, `R5`, and `R6` behave similarly, although the evidence profile is different. `R7` and `R8` remain ambiguous but comparatively isolated.

This is where the governance lens becomes practical. The point is not to invent a mystical notion of fractality. The point is to identify record groups whose behavior is unstable across scale or threshold and therefore deserve human attention.

## Why this belongs after the MDM primer

Without MDM language, the threshold plot is just a technical chart. With MDM language, it becomes a stewardship problem:

- Which cluster is stable enough for automatic merge?
- Which cluster changes too easily and should go to a queue?
- Which reference attributes should have higher weight before merge logic is trusted?

## Exercise

1. change the thresholds and inspect how the instability table changes
2. add a weighted survivorship rule for email or city
3. define a stewardship policy: what level of instability should force human review?
